# 01 - Text extraction

**Goal.** Walk the first stage of the ingestion pipeline on small sample files: pre-parse validation
guardrails, extraction into structural blocks, cleaning, language detection and metadata inference.

**What it checks** (every check ends with an `assert`, so the notebook doubles as a test):
- good files pass `validate_file`; spoofed, empty, macro, encrypted and JavaScript files are rejected with a stable error code
- PDF (with a table), DOCX, PPTX, HTML, Markdown and TXT (Urdu) extract into blocks with page numbers and section paths
- a PDF table becomes a Markdown table block
- repeated page headers and "Page N of M" footers are removed
- a scanned PDF has no text layer; with an OCR engine its page becomes an OCR block
- language, agency, jurisdiction and effective date are inferred (Urdu digits included)

**How to run.** From the repo root, with the project venv:
`.venv/Scripts/python.exe scripts/run_notebooks.py` (all notebooks, executed in place), or open this
notebook with the `rag-venv` kernel. No network or API key is needed. See `notebooks/README.md`.

All files are built into a temp directory with `tests/fixtures/files/factories.py`; paths below are relative to it.

In [1]:
import warnings

warnings.filterwarnings("ignore")
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "app").is_dir() else Path.cwd().parent
sys.path[:0] = [str(ROOT), str(ROOT / "notebooks")]

from collections import Counter

from _support import FakeOcr, clip, make_samples, new_workdir, notebook_settings, rel
from app.core.errors import AppError
from app.ingestion.cleaning import clean_blocks, join_blocks
from app.ingestion.extractors import default_extractors
from app.ingestion.extractors.pdf import PdfExtractor
from app.ingestion.language import detect_language, detect_query_language, normalize_digits
from app.ingestion.metadata import find_effective_date, infer_agency, infer_jurisdiction, infer_topic
from app.ingestion.validation import validate_file
from tests.fixtures.files import factories as f

settings = notebook_settings()
work = new_workdir()
samples = make_samples(work)
for s in samples.values():
    print(f"{rel(s.path, work):28} {s.path.stat().st_size:>7} bytes")

passport_fee_table.pdf         72709 bytes
passport_guide.pdf              2982 bytes
nadra_cnic_guide.docx          37582 bytes
fbr_tax_slides.pptx            29358 bytes
sehat_card_faq.html              600 bytes
consumer_complaint.md            335 bytes
passport_fee_urdu.txt            442 bytes
scanned_form.pdf                 668 bytes
helpdesk_note.txt                171 bytes


## 1. Validation guardrails (GD-1 .. GD-8)
`validate_file` runs before any parser touches a file. Accepted files return their canonical type;
rejected ones raise `AppError` with a stable code.

In [2]:
accepted = {}
for key, s in samples.items():
    accepted[key] = validate_file(s.path, s.name, settings)
print(accepted)
assert accepted["pdf_table"] == "pdf" and accepted["docx"] == "docx" and accepted["urdu"] == "txt"

bad = work / "bad"
bad.mkdir()
(bad / "empty.pdf").write_bytes(b"")
(bad / "spoofed.pdf").write_bytes(b"MZ\x90\x00 this is a Windows executable renamed to .pdf " * 20)
f.make_macro_docx(bad / "macro.docx")
f.make_encrypted_pdf(bad / "locked.pdf")
f.make_js_pdf(bad / "active.pdf")
(bad / "script.exe").write_bytes(b"MZ")

expected = {
    "empty.pdf": "empty_file",
    "spoofed.pdf": "unsupported_media_type",
    "macro.docx": "active_content_rejected",
    "locked.pdf": "encrypted_document",
    "active.pdf": "active_content_rejected",
    "script.exe": "unsupported_media_type",
}
print(f"{'file':14} {'HTTP':>4}  code")
for name, want in expected.items():
    try:
        validate_file(bad / name, name, settings)
    except AppError as exc:
        print(f"{name:14} {exc.code.status:>4}  {exc.code.slug}")
        assert exc.code.slug == want, (name, exc.code.slug)
    else:
        raise AssertionError(f"{name} was NOT rejected")

{'pdf_table': 'pdf', 'pdf_pages': 'pdf', 'docx': 'docx', 'pptx': 'pptx', 'html': 'html', 'md': 'md', 'urdu': 'txt', 'scan': 'pdf', 'private': 'txt'}
file           HTTP  code
empty.pdf       400  empty_file
spoofed.pdf     415  unsupported_media_type


macro.docx      400  active_content_rejected
locked.pdf      400  encrypted_document
active.pdf      400  active_content_rejected
script.exe      415  unsupported_media_type


## 2. Extract every file type
Each extractor returns `ExtractedBlock`s (paragraph, heading, table, list, ocr) with page numbers
(slide numbers for PPTX) and heading paths. `clean_blocks` then normalises the text.

In [3]:
extractors = default_extractors()
docs = {}
print(f"{'sample':10} {'type':5} {'pages':>5} {'blocks':>6}  kinds")
for key, s in samples.items():
    if key in ("scan", "private"):
        continue
    file_type = validate_file(s.path, s.name, settings)
    extracted = extractors[file_type].extract(s.path)
    blocks = clean_blocks(extracted.blocks)
    docs[key] = (extracted, blocks)
    kinds = dict(Counter(b.kind for b in blocks))
    print(f"{key:10} {file_type:5} {extracted.page_count:>5} {len(blocks):>6}  {kinds}")
    assert blocks, key


def show(key: str, limit: int = 6) -> None:
    extracted, blocks = docs[key]
    print(f"-- {key}: title={clip(extracted.title or '-', 60)!r}")
    for b in blocks[:limit]:
        print(f"   p{b.page} {b.kind:9} {' > '.join(b.section_path) or '-':30.30} {clip(b.text, 70)!r}")


for key in ("docx", "pptx", "html", "md"):
    show(key)

sample     type  pages blocks  kinds


pdf_table  pdf       1      2  {'paragraph': 1, 'table': 1}
pdf_pages  pdf       2      4  {'paragraph': 4}


docx       docx      1     13  {'heading': 5, 'paragraph': 7, 'table': 1}
pptx       pptx      2      4  {'heading': 2, 'paragraph': 1, 'table': 1}
html       html      1      6  {'heading': 3, 'paragraph': 2, 'table': 1}
md         md        1      5  {'heading': 3, 'paragraph': 2}
urdu       txt       1      3  {'paragraph': 3}
-- docx: title='NADRA CNIC Guide'
   pNone heading   NADRA CNIC Guide               'NADRA CNIC Guide'
   pNone heading   NADRA CNIC Guide > Eligibility 'Eligibility'
   pNone paragraph NADRA CNIC Guide > Eligibility 'NADRA issues the computerised national identity card to every citizen…'
   pNone paragraph NADRA CNIC Guide > Eligibility 'A juvenile card (the Smart Card for children) is available for citize…'
   pNone heading   NADRA CNIC Guide > Renewal     'Renewal'
   pNone paragraph NADRA CNIC Guide > Renewal     'A citizen can renew the identity card at any NADRA registration centr…'
-- pptx: title='FBR income tax filing'
   p1 heading   FBR income tax fi

## 3. PDF table as Markdown

In [4]:
_, blocks = docs["pdf_table"]
tables = [b for b in blocks if b.kind == "table"]
print(tables[0].text)
assert len(tables) == 1 and tables[0].text.startswith("| Category | Pages | Fee |")
assert "Rs 4,500" in tables[0].text and "Rs 9,000" in tables[0].text

_, docx_blocks = docs["docx"]
docx_table = next(b for b in docx_blocks if b.kind == "table")
print(docx_table.text)
assert docx_table.text.splitlines()[0].startswith("| Category")

| Category | Pages | Fee |
|---|---|---|
| Normal | 36 | Rs 4,500 |
| Urgent | 36 | Rs 9,000 |
| Category | Processing time | Fee |
|---|---|---|
| Normal | 15 days | Rs 750 |
| Urgent | 7 days | Rs 1,500 |
| Executive | 2 days | Rs 2,500 |


## 4. Header / footer cleaning and page numbers
The PDF has the running header "Immigration and Passports - Official Guide" and a "Page N of 2"
footer on both pages. Both are detected as repeated margin lines and dropped; blocks keep their page number.

In [5]:
extracted, blocks = docs["pdf_pages"]
text = join_blocks(blocks)
for b in blocks:
    print(f"p{b.page} {b.kind:9} {clip(b.text, 90)!r}")
assert "Official Guide" not in text, "running header should be removed"
assert "Page 1 of 2" not in text and "Page 2 of 2" not in text, "page-number footer should be removed"
assert {b.page for b in blocks} == {1, 2}
assert extracted.page_count == 2

p1 paragraph 'Immigration and Passports: Machine Readable Passport guide. Date of issue: 12 March 2025.…'
p1 paragraph 'The applicant must submit the national identity card, two photographs and the fee challan…'
p2 paragraph 'The fee schedule depends on the number of pages and on the category. Urgent passport fee …'
p2 paragraph 'A passport that has been lost must be reported to the police. The applicant then applies …'


## 5. Scanned PDF: no text layer, then OCR
Without an OCR engine the page yields no blocks (the ingestion service turns that into
`no_extractable_text`). With an engine, the page becomes an `ocr` block with a confidence.
A fake engine stands in for Tesseract here, as in the unit tests.

In [6]:
scan = samples["scan"].path
no_ocr = PdfExtractor().extract(scan)
print("without OCR:", len(no_ocr.blocks), "blocks, ocr_pages =", no_ocr.ocr_pages)
assert len(no_ocr.blocks) == 0

with_ocr = PdfExtractor(ocr=FakeOcr()).extract(scan)
ocr_blocks = clean_blocks(with_ocr.blocks)
print("with OCR   :", len(ocr_blocks), "block(s), ocr_pages =", with_ocr.ocr_pages)
for b in ocr_blocks:
    print(f"   p{b.page} kind={b.kind} is_ocr={b.is_ocr} confidence={b.ocr_confidence} {clip(b.text, 60)!r}")
assert with_ocr.ocr_pages == 1 and ocr_blocks and all(b.is_ocr for b in ocr_blocks)

without OCR: 0 blocks, ocr_pages = 0
with OCR   : 1 block(s), ocr_pages = 1
   p1 kind=ocr is_ocr=True confidence=90.0 'The applicant shall submit the application form at the regi…'


## 6. Urdu text and language detection
Cleaning keeps the original (Urdu) digits for display; `normalize_digits` maps them to ASCII for
hashing and the sparse index so amounts match across scripts.

In [7]:
extracted, blocks = docs["urdu"]
for b in blocks:
    print(f"{detect_language(b.text)}  {clip(b.text, 80)!r}")
sentence = "عام پاسپورٹ کی فیس ۴۵۰۰ روپے ہے"
print("display :", sentence)
print("indexed :", normalize_digits(sentence))
assert all(detect_language(b.text) == "ur" for b in blocks)
assert "4500" in normalize_digits(sentence)
assert detect_language(docs["docx"][1][1].text) == "en"
queries = {
    "What is the urgent passport fee?": "en",
    "پاسپورٹ کی فیس کتنی ہے؟": "ur",
    "Urgent passport banwane ki fee kitni hai?": "roman_ur",
}
for q, want in queries.items():
    got = detect_query_language(q)
    print(f"{got:9} {q}")
    assert got == want

ur  'پاسپورٹ کی فیس'
ur  'عام پاسپورٹ کی فیس ۴۵۰۰ روپے ہے۔ فوری پاسپورٹ کی فیس ۹۰۰۰ روپے ہے۔ درخواست تمام…'
ur  'یہ معلومات ڈائریکٹوریٹ جنرل امیگریشن اینڈ پاسپورٹس کی طرف سے جاری کی گئی ہیں۔ ت…'
display : عام پاسپورٹ کی فیس ۴۵۰۰ روپے ہے
indexed : عام پاسپورٹ کی فیس 4500 روپے ہے
en        What is the urgent passport fee?
ur        پاسپورٹ کی فیس کتنی ہے؟
roman_ur  Urgent passport banwane ki fee kitni hai?


## 7. Inferred metadata (agency, jurisdiction, topic, effective date)
For uploads these come from the file name and the first part of the text (a PDF's page header date wins).

In [8]:
print(f"{'sample':10} {'agency':9} {'juris.':8} {'topic':20} effective date")
meta = {}
for key, (extracted, blocks) in docs.items():
    head = join_blocks(blocks)[:20_000]
    name = samples[key].name
    agency = infer_agency(name, head)
    effective = extracted.metadata.get("effective_date") or find_effective_date(head)
    meta[key] = (agency, infer_jurisdiction(agency, head), infer_topic(name, head), effective)
    a, j, t, d = meta[key]
    print(f"{key:10} {str(a):9} {str(j):8} {str(t):20} {d}")

assert meta["pdf_table"][:3] == ("DGIP", "federal", "passport") and str(meta["pdf_table"][3]) == "2025-03-12"
assert meta["docx"][:2] == ("NADRA", "federal")
assert meta["html"][:2] == ("KP Sehat", "kp")
assert meta["md"][:2] == ("Punjab", "punjab")
assert str(meta["urdu"][3]) == "2026-09-18", "Urdu digits and month name should parse"

sample     agency    juris.   topic                effective date
pdf_table  DGIP      federal  passport             2025-03-12
pdf_pages  DGIP      federal  passport             2025-03-12
docx       NADRA     federal  identity             None
pptx       FBR       federal  tax                  None
html       KP Sehat  kp       health               None
md         Punjab    punjab   consumer_protection  None
urdu       DGIP      federal  passport             2026-09-18


## 8. HTML: scripts are stripped, never executed

In [9]:
_, blocks = docs["html"]
html_text = join_blocks(blocks)
assert "never executed" not in html_text and "alert(" not in html_text
assert any(b.kind == "table" for b in blocks)
print("HTML script removed; table kept:", [b.kind for b in blocks])
print("\nAll extraction checks passed.")

HTML script removed; table kept: ['heading', 'heading', 'paragraph', 'heading', 'paragraph', 'table']

All extraction checks passed.
